# Day 8 — Exploratory Time-Series Forecasting

## Objective
Explore the weekly distribution of recorded `Last_Login_Date` values and produce an illustrative 90-day (13-week) forecast with prediction intervals.

**Important limitation:** the dataset has one `Last_Login_Date` per student, not repeated weekly login-event history. Therefore this analysis forecasts the *number of student records whose recorded last-login date falls in each week*; it does **not** establish future LMS activity, student retention, GPA, or probation trends. Treat results as an exploratory demonstration only.


## 1. Load the cleaned dataset
Keep this notebook in `Day_8_Forecasting/` so the relative path below points to the existing `Dataset/` folder.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import t

DATA_PATH = "../Dataset/Cleaned_Academic_Probation_Project.csv"
if not os.path.exists(DATA_PATH):
    # Also allow running the notebook from the repository root.
    DATA_PATH = "Dataset/Cleaned_Academic_Probation_Project.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df["Last_Login_Date"] = pd.to_datetime(df["Last_Login_Date"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["Last_Login_Date"]).copy()

print("Rows with a valid Last_Login_Date:", len(df))
print("Earliest recorded date:", df["Last_Login_Date"].min().date())
print("Latest recorded date:", df["Last_Login_Date"].max().date())
df[["Student_ID", "Last_Login_Date"]].head()

## 2. Create a weekly series
Each student contributes one record to the week containing their recorded last-login date. Missing dates are excluded.

In [ ]:
weekly = (
    df.set_index("Last_Login_Date")
      .resample("W-SUN")
      .size()
      .asfreq("W-SUN", fill_value=0)
      .rename("Recorded_Last_Login_Records")
)

if len(weekly) < 8:
    raise ValueError("Not enough weekly observations for this exploratory forecast. Check the date column and dataset.")

print("Number of weekly observations:", len(weekly))
display(weekly.to_frame().tail(10))

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(weekly.index, weekly.values, marker="o", linewidth=1.5)
plt.title("Weekly Count of Recorded Last-Login Dates")
plt.xlabel("Week ending")
plt.ylabel("Number of student records")
plt.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs("Output_Charts", exist_ok=True)
plt.savefig("Output_Charts/weekly_recorded_last_login_counts.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Fit a simple trend model
A linear trend is used as a transparent baseline. Prediction intervals below are regression-based intervals; with a short and potentially synthetic snapshot, they should be interpreted cautiously.

In [ ]:
y = weekly.to_numpy(dtype=float)
x = np.arange(len(y), dtype=float).reshape(-1, 1)
model = LinearRegression().fit(x, y)
fitted = model.predict(x)
residuals = y - fitted
n = len(y)
p = 2  # intercept and slope
df_resid = n - p
if df_resid <= 0:
    raise ValueError("Not enough observations to estimate prediction intervals.")
residual_sd = np.sqrt(np.sum(residuals ** 2) / df_resid)
mse = np.sum(residuals ** 2) / df_resid
xbar = x.mean()
Sxx = np.sum((x.ravel() - xbar) ** 2)


## 4. Forecast the next 90 days
Thirteen weekly steps approximate 90 days. The forecast starts after the final week available in the dataset, not after today's date.

In [ ]:
horizon = 13
x_future = np.arange(n, n + horizon, dtype=float).reshape(-1, 1)
pred = model.predict(x_future)
se_pred = residual_sd * np.sqrt(1 + (1 / n) + ((x_future.ravel() - xbar) ** 2 / Sxx))
critical_t = t.ppf(0.975, df_resid)
lower = np.maximum(0, pred - critical_t * se_pred)
upper = np.maximum(lower, pred + critical_t * se_pred)
future_dates = pd.date_range(start=weekly.index[-1] + pd.Timedelta(days=7), periods=horizon, freq="W-SUN")

forecast = pd.DataFrame({
    "Week_Ending": future_dates,
    "Forecast_Records": np.maximum(0, pred),
    "Lower_95_Prediction_Interval": lower,
    "Upper_95_Prediction_Interval": upper,
})
forecast.to_csv("Output_Charts/90_day_weekly_forecast.csv", index=False)
display(forecast.round(2))

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(weekly.index, weekly.values, label="Observed weekly record counts", marker="o", linewidth=1.5)
plt.plot(future_dates, pred, label="Linear-trend forecast", marker="o", linestyle="--")
plt.fill_between(future_dates, lower, upper, alpha=0.2, label="Approx. 95% prediction interval")
plt.axvline(weekly.index[-1], linestyle=":", linewidth=1, label="Forecast start")
plt.title("Exploratory 90-Day Forecast of Recorded Last-Login-Date Counts")
plt.xlabel("Week ending")
plt.ylabel("Number of student records (forecast may be fractional)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("Output_Charts/90_day_recorded_last_login_forecast.png", dpi=200, bbox_inches="tight")
plt.show()

print("Forecast begins:", future_dates.min().date())
print("Forecast ends:", future_dates.max().date())
print("Mean forecast per week:", round(float(np.mean(pred)), 2))

## 5. Interpretation and limitations
- This notebook demonstrates a simple time-series-style workflow using the date field available in the current project dataset.
- The weekly counts describe the distribution of one recorded last-login date per student. They are not repeated observations of logins over time.
- The linear trend and prediction intervals are illustrative and may be misleading if the dates are synthetic, irregularly collected, or not generated from a continuing process.
- For a valid operational forecast, collect timestamped login events or weekly/monthly academic KPI snapshots across multiple periods, then evaluate forecasts using a chronological holdout set and suitable error metrics.
- Do not use this forecast to make individual student decisions.